# 01. Data Overview

## 팀 합의 기준 v1 데이터 점검

이 노트북은 최초 raw 파일을 검토하는 노트북이 아니라, 팀 논의를 거쳐 정리된 `v1` 기준 데이터를 점검하는 노트북입니다.

따라서 이 노트북의 기준 입력은 `Membership_v1.csv`, `User_Mapping_v1.csv`, `View_History_v1.csv`, `Movie_Master_v1.csv`입니다.  
Wavve/KOBIS 영화 메타데이터는 별도 원천 파일로 남아 있으므로, 가능한 경우 `_data/01_raw/`에서 함께 읽어 커버리지와 연결 가능성을 확인합니다.

핵심 목적은 다음과 같습니다.

1. 팀 합의 기준 v1 파일이 정상적으로 로딩되는지 확인한다.
2. 각 파일의 행 수, 컬럼, key 구조를 확인한다.
3. `price == 100`과 `is_promotion == 1`의 관계를 확인한다.
4. 더미 이상치 후보가 v1 기준에 남아 있는지 확인한다.
5. `USER_KEY -> USER_NUM -> View_History` 연결 구조를 확인한다.
6. `View_History.MOVIE_NUM -> Movie_Master_v1.MOVIE_NUM` 연결 가능성을 확인한다.
7. Wavve/KOBIS 메타데이터가 View_History 등장 영화 기준으로 어느 정도 커버하는지 예비 점검한다.

이 노트북은 데이터를 제거하거나 파생변수를 생성하지 않습니다. 전처리 정책은 02번 노트북에서 다룹니다.

## 1-1. 입력 파일과 출력 파일

### 기준 입력 파일

권장 위치는 다음입니다.

```text
_data/02_interim/260430_membership_v1(이상치, 이름 변경)/
├─ Membership_v1.csv
├─ User_Mapping_v1.csv
├─ View_History_v1.csv
└─ Movie_Master_v1.csv
```

영화 메타데이터 원천 파일은 보통 다음 위치에 있다고 가정합니다.

```text
_data/01_raw/
├─ Wavve_movie(Regex).csv
└─ Wavve_movie(KOBIS).csv
```

파일명이 다를 가능성을 고려해 노트북 안에서 후보 파일명을 자동 탐색합니다.

### 출력 파일

```text
reports/tables/01_data_overview_file_summary.csv
reports/tables/01_data_overview_column_overview.csv
reports/tables/01_data_overview_key_checks.csv
reports/tables/01_data_overview_metadata_coverage.csv
reports/tables/01_data_overview_membership_basic_summary.csv
```

In [22]:
from pathlib import Path
import re
import json
import warnings

import numpy as np
import pandas as pd
from IPython.display import display

pd.set_option("display.max_columns", 120)
pd.set_option("display.width", 180)
warnings.filterwarnings("ignore", category=FutureWarning)

## 1-2. 경로 자동 탐색

이 노트북은 `ott-churn-prediction` 루트에서 실행해도 되고, `park.ingyeom` 폴더 안에서 실행해도 됩니다.  
현재 작업 디렉터리 또는 상위 폴더에서 `_data` 폴더를 찾아 프로젝트 루트를 결정합니다.

In [23]:
def find_project_root(start: Path | None = None) -> Path:
    start = Path.cwd() if start is None else Path(start)
    candidates = [start, *start.parents]

    for candidate in candidates:
        if (candidate / ".git").exists() and (candidate / "_data").exists():
            return candidate

    for candidate in candidates:
        interim_dir = candidate / "_data" / "02_interim"
        if interim_dir.exists() and any(interim_dir.rglob("Membership_v1.csv")):
            return candidate

    for candidate in candidates:
        if (candidate / "_data").exists():
            return candidate

    return start

PROJECT_ROOT = find_project_root()

DATA_DIR = PROJECT_ROOT / "_data"
RAW_DIR = DATA_DIR / "01_raw"
INTERIM_DIR = DATA_DIR / "02_interim"

WORK_ROOT = PROJECT_ROOT / "park.ingyeom"
REPORTS_DIR = WORK_ROOT / "reports"
TABLES_DIR = REPORTS_DIR / "tables"

TABLES_DIR.mkdir(parents=True, exist_ok=True)

print("PROJECT_ROOT:", PROJECT_ROOT)
print("DATA_DIR:", DATA_DIR)
print("RAW_DIR:", RAW_DIR)
print("INTERIM_DIR:", INTERIM_DIR)
print("WORK_ROOT:", WORK_ROOT)
print("TABLES_DIR:", TABLES_DIR)

PROJECT_ROOT: c:\Code\ott-churn-prediction
DATA_DIR: c:\Code\ott-churn-prediction\_data
RAW_DIR: c:\Code\ott-churn-prediction\_data\01_raw
INTERIM_DIR: c:\Code\ott-churn-prediction\_data\02_interim
WORK_ROOT: c:\Code\ott-churn-prediction\park.ingyeom
TABLES_DIR: c:\Code\ott-churn-prediction\park.ingyeom\reports\tables


## 1-3. v1 기준 데이터 폴더 탐색

`Membership_v1.csv`, `User_Mapping_v1.csv`, `View_History_v1.csv`, `Movie_Master_v1.csv` 네 파일이 모두 있는 폴더를 찾습니다.  
폴더명에 `260430` 또는 `membership_v1`이 들어간 경로를 우선합니다.

In [24]:
V1_REQUIRED_FILES = [
    "Membership_v1.csv",
    "User_Mapping_v1.csv",
    "View_History_v1.csv",
    "Movie_Master_v1.csv",
]


def has_all_files(folder: Path, file_names: list[str]) -> bool:
    return folder.exists() and all((folder / name).exists() for name in file_names)


def find_v1_base_dir() -> Path:
    candidates = []
    candidates.append(INTERIM_DIR / "260430_membership_v1(이상치, 이름 변경)")

    if INTERIM_DIR.exists():
        candidates.extend([p for p in INTERIM_DIR.rglob("*") if p.is_dir()])

    if RAW_DIR.exists():
        candidates.append(RAW_DIR)

    candidates.append(Path("/mnt/data"))

    valid = [p for p in candidates if has_all_files(p, V1_REQUIRED_FILES)]
    if not valid:
        searched = "\n".join(str(p) for p in candidates[:30])
        raise FileNotFoundError(
            "v1 기준 파일 4개가 모두 있는 폴더를 찾지 못했습니다.\n"
            "필요 파일: " + ", ".join(V1_REQUIRED_FILES) + "\n"
            "검색 후보 예시:\n" + searched
        )

    def score(path: Path) -> tuple[int, str]:
        s = str(path).lower()
        priority = 0
        if "260430" in s:
            priority += 10
        if "membership_v1" in s:
            priority += 5
        if "02_interim" in s:
            priority += 3
        if "01_raw" in s:
            priority -= 5
        return (-priority, str(path))

    return sorted(valid, key=score)[0]

V1_BASE_DIR = find_v1_base_dir()
print("V1_BASE_DIR:", V1_BASE_DIR)

if "01_raw" in str(V1_BASE_DIR):
    print("WARNING: v1 파일이 01_raw 안에서 발견되었습니다. 프로젝트 구조상 02_interim의 팀 합의 기준 폴더에 두는 것을 권장합니다.")

V1_BASE_DIR: c:\Code\ott-churn-prediction\_data\02_interim\260430_membership_v1(이상치, 이름변경)


## 1-4. 영화 메타데이터 원천 파일 탐색

Wavve/KOBIS 파일은 팀 합의 v1 폴더가 아니라 `_data/01_raw`에 있을 수 있습니다.  
파일명이 공백, 괄호, 한글에 따라 달라질 수 있으므로 후보명을 순서대로 탐색합니다.

In [25]:
METADATA_FILE_CANDIDATES = {
    "wavve": [
        "Wavve_movie(Regex).csv",
        "wavve_movies_filtered_by  정규식 (1)(1).csv",
        "wavve_movies_filtered_by_정규식.csv",
        "Wavve_movie(Regex).CSV",
    ],
    "kobis": [
        "Wavve_movie(KOBIS).csv",
        "wavve_notfound_kobis_filtered_by_char_match(1).csv",
        "wavve_notfound_kobis_filtered_by_char_match.csv",
        "Wavve_movie(KOBIS).CSV",
    ],
}


def find_file_by_candidates(search_dirs: list[Path], candidates: list[str]) -> Path | None:
    for folder in search_dirs:
        if not folder.exists():
            continue
        for name in candidates:
            p = folder / name
            if p.exists():
                return p
    return None

METADATA_SEARCH_DIRS = [RAW_DIR, V1_BASE_DIR, PROJECT_ROOT, Path("/mnt/data")]

PATHS = {
    "membership": V1_BASE_DIR / "Membership_v1.csv",
    "mapping": V1_BASE_DIR / "User_Mapping_v1.csv",
    "view": V1_BASE_DIR / "View_History_v1.csv",
    "movie_master": V1_BASE_DIR / "Movie_Master_v1.csv",
    "wavve": find_file_by_candidates(METADATA_SEARCH_DIRS, METADATA_FILE_CANDIDATES["wavve"]),
    "kobis": find_file_by_candidates(METADATA_SEARCH_DIRS, METADATA_FILE_CANDIDATES["kobis"]),
}

for key, path in PATHS.items():
    print(f"{key:>12}:", path)

missing_required = [k for k in ["membership", "mapping", "view", "movie_master"] if PATHS[k] is None or not Path(PATHS[k]).exists()]
if missing_required:
    raise FileNotFoundError(f"필수 v1 파일을 찾지 못했습니다: {missing_required}")

  membership: c:\Code\ott-churn-prediction\_data\02_interim\260430_membership_v1(이상치, 이름변경)\Membership_v1.csv
     mapping: c:\Code\ott-churn-prediction\_data\02_interim\260430_membership_v1(이상치, 이름변경)\User_Mapping_v1.csv
        view: c:\Code\ott-churn-prediction\_data\02_interim\260430_membership_v1(이상치, 이름변경)\View_History_v1.csv
movie_master: c:\Code\ott-churn-prediction\_data\02_interim\260430_membership_v1(이상치, 이름변경)\Movie_Master_v1.csv
       wavve: c:\Code\ott-churn-prediction\_data\01_raw\Wavve_movie(Regex).csv
       kobis: c:\Code\ott-churn-prediction\_data\01_raw\Wavve_movie(KOBIS).csv


## 1-5. 데이터 로딩

CSV 인코딩은 `utf-8-sig`, `utf-8`, `cp949`, `euc-kr` 순서로 시도합니다.

In [26]:
def read_csv_safely(path: Path | None) -> pd.DataFrame | None:
    if path is None:
        return None
    path = Path(path)
    if not path.exists():
        return None

    encodings = ["utf-8-sig", "utf-8", "cp949", "euc-kr"]
    last_error = None
    for enc in encodings:
        try:
            return pd.read_csv(path, encoding=enc)
        except UnicodeDecodeError as e:
            last_error = e
    raise last_error

membership = read_csv_safely(PATHS["membership"])
mapping = read_csv_safely(PATHS["mapping"])
view = read_csv_safely(PATHS["view"])
movie_master = read_csv_safely(PATHS["movie_master"])
wavve = read_csv_safely(PATHS["wavve"])
kobis = read_csv_safely(PATHS["kobis"])

loaded = {
    "membership": membership,
    "mapping": mapping,
    "view": view,
    "movie_master": movie_master,
    "wavve": wavve,
    "kobis": kobis,
}

file_summary = []
for name, df in loaded.items():
    path = PATHS.get(name)
    file_summary.append({
        "dataset": name,
        "path": str(path) if path is not None else None,
        "loaded": df is not None,
        "rows": None if df is None else len(df),
        "columns": None if df is None else df.shape[1],
    })
file_summary = pd.DataFrame(file_summary)
display(file_summary)
file_summary.to_csv(TABLES_DIR / "01_data_overview_file_summary.csv", index=False, encoding="utf-8-sig")

,dataset,path,loaded,rows,columns
0,membership,c:\Code\ott-churn-prediction\_data\02_interim\...,True,17876,15
1,mapping,c:\Code\ott-churn-prediction\_data\02_interim\...,True,19877,2
2,view,c:\Code\ott-churn-prediction\_data\02_interim\...,True,106205,5
3,movie_master,c:\Code\ott-churn-prediction\_data\02_interim\...,True,14018,3
4,wavve,c:\Code\ott-churn-prediction\_data\01_raw\Wavv...,True,4060,31
5,kobis,c:\Code\ott-churn-prediction\_data\01_raw\Wavv...,True,999,13


## 1-6. 컬럼 구조 확인

In [27]:
column_overview = []
for name, df in loaded.items():
    if df is None:
        continue
    for idx, col in enumerate(df.columns):
        column_overview.append({
            "dataset": name,
            "column_order": idx,
            "column_name": col,
            "dtype": str(df[col].dtype),
            "n_missing": int(df[col].isna().sum()),
            "missing_rate": float(df[col].isna().mean()),
            "n_unique": int(df[col].nunique(dropna=True)),
        })
column_overview = pd.DataFrame(column_overview)
display(column_overview.head(80))
column_overview.to_csv(TABLES_DIR / "01_data_overview_column_overview.csv", index=False, encoding="utf-8-sig")

,dataset,column_order,column_name,dtype,n_missing,missing_rate,n_unique
0,membership,0,USER_KEY,object,0,0.000000,17578
1,membership,1,product_code,object,0,0.000000,40
2,membership,2,price,float64,0,0.000000,29
3,membership,3,billing_method,int64,0,0.000000,10
4,membership,4,max_screen,int64,0,0.000000,3
...,...,...,...,...,...,...,...
64,kobis,8,genreNm,object,7,0.007007,230
65,kobis,9,directors,object,95,0.095095,777
66,kobis,10,actors,object,154,0.154154,826
67,kobis,11,watchGrade,object,170,0.170170,18


## 1-7. 컬럼 alias 생성

v1 기준 파일과 원본 파일 간 컬럼명이 일부 다를 수 있습니다.  
01번에서는 원본 파일을 변경하지 않고, 검산을 위해 메모리 안에서만 표준 alias를 사용합니다.

In [28]:
def strip_columns(df: pd.DataFrame | None) -> pd.DataFrame | None:
    if df is None:
        return None
    out = df.copy()
    out.columns = [str(c).strip() for c in out.columns]
    return out

membership = strip_columns(membership)
mapping = strip_columns(mapping)
view = strip_columns(view)
movie_master = strip_columns(movie_master)
wavve = strip_columns(wavve)
kobis = strip_columns(kobis)


def standardize_movie_master_for_check(df: pd.DataFrame) -> pd.DataFrame:
    out = df.copy()
    out = out.rename(columns={
        "MOVIE_ID": "MOVIE_NUM",
        "TITLE": "movie_title",
        "RELEASE_MONTH": "ott_release_month",
    })
    return out


def standardize_view_for_check(df: pd.DataFrame) -> pd.DataFrame:
    out = df.copy()
    out = out.rename(columns={
        "watch_time(min)": "watch_time",
        "WATCH_TIME": "watch_time",
        "WATCH_DAY": "watch_day",
        "USER_ID": "USER_NUM",
    })
    return out

movie_master_check = standardize_movie_master_for_check(movie_master)
view_check = standardize_view_for_check(view)

print("Movie_Master columns after in-memory alias:")
print(movie_master_check.columns.tolist())
print("\nView_History columns after in-memory alias:")
print(view_check.columns.tolist())

Movie_Master columns after in-memory alias:
['MOVIE_NUM', 'movie_title', 'ott_release_month']

View_History columns after in-memory alias:
['USER_NUM', 'MOVIE_NUM', 'watch_time', 'watch_day', 'watch_seq']


## 1-8. Membership 기본 점검

전처리를 하지 않은 상태에서 target 분포, 100원딜 정의, 더미 이상치 후보를 확인합니다.

In [29]:
required_membership_cols = ["USER_KEY", "is_repurchase"]
missing_cols = [c for c in required_membership_cols if c not in membership.columns]
if missing_cols:
    raise KeyError(f"Membership 필수 컬럼이 없습니다: {missing_cols}")

membership_basic = {
    "rows": len(membership),
    "unique_USER_KEY": membership["USER_KEY"].nunique(dropna=True),
    "duplicated_USER_KEY_rows": int(membership["USER_KEY"].duplicated(keep=False).sum()),
    "target_missing": int(membership["is_repurchase"].isna().sum()),
    "target_mean": float(membership["is_repurchase"].mean()) if pd.api.types.is_numeric_dtype(membership["is_repurchase"]) else None,
}

membership_basic_summary = pd.DataFrame([membership_basic])
display(membership_basic_summary)
membership_basic_summary.to_csv(TABLES_DIR / "01_data_overview_membership_basic_summary.csv", index=False, encoding="utf-8-sig")

print("is_repurchase distribution:")
display(membership["is_repurchase"].value_counts(dropna=False).rename("count").to_frame())

,rows,unique_USER_KEY,duplicated_USER_KEY_rows,target_missing,target_mean
0,17876,17578,564,0,0.661837


is_repurchase distribution:


,count
is_repurchase,
1,11831
0,6045


In [30]:
if {"price", "is_promotion"}.issubset(membership.columns):
    membership_price_check = membership.copy()
    membership_price_check["price_100_flag"] = (membership_price_check["price"] == 100).astype(int)
    promo_price_crosstab = pd.crosstab(
        membership_price_check["is_promotion"],
        membership_price_check["price_100_flag"],
        rownames=["is_promotion"],
        colnames=["price_100_flag"],
        dropna=False,
    )
    display(promo_price_crosstab)

    if "is_repurchase" in membership_price_check.columns:
        repurchase_by_100 = (
            membership_price_check
            .groupby("price_100_flag")["is_repurchase"]
            .agg(["count", "mean"])
            .rename(index={0: "not_100won", 1: "100won"})
        )
        display(repurchase_by_100)
else:
    print("price 또는 is_promotion 컬럼이 없어 100원딜 관계 검산을 건너뜁니다.")

price_100_flag,0,1
is_promotion,,
0,8815,0
1,0,9061


,count,mean
price_100_flag,,
not_100won,8815,0.698355
100won,9061,0.626311


In [31]:
dummy_cols = {"gender", "is_user_verified", "age"}
if dummy_cols.issubset(membership.columns):
    dummy_mask = (
        (membership["gender"] == "N")
        & (membership["is_user_verified"] == 0)
        & (membership["age"] == 40)
    )
    print("dummy anomaly candidate rows:", int(dummy_mask.sum()))
    if "is_promotion" in membership.columns:
        display(
            membership.assign(dummy_anomaly_candidate=dummy_mask.astype(int))
            .groupby(["dummy_anomaly_candidate", "is_promotion"])["is_repurchase"]
            .agg(["count", "mean"])
        )
else:
    print("gender/is_user_verified/age 컬럼 중 일부가 없어 더미 이상치 후보 검산을 건너뜁니다.")

dummy anomaly candidate rows: 2639


count      mean
dummy_anomaly_candidate is_promotion                 
0                       0              6176  0.712759
                        1              9061  0.626311
1                       0              2639  0.664646

## 1-9. User_Mapping key 구조 확인

동일 `USER_KEY`가 여러 `USER_NUM`과 연결될 수 있습니다.  
현재 프로젝트는 구독 이벤트 단위 분석을 기본으로 하므로, 이 현상은 즉시 제거 대상이 아니라 이후 02번에서 정책으로 명시해야 합니다.

In [32]:
required_mapping_cols = ["USER_KEY", "USER_NUM"]
missing_cols = [c for c in required_mapping_cols if c not in mapping.columns]
if missing_cols:
    raise KeyError(f"User_Mapping 필수 컬럼이 없습니다: {missing_cols}")

mapping_summary = pd.DataFrame([{
    "rows": len(mapping),
    "unique_USER_KEY": mapping["USER_KEY"].nunique(dropna=True),
    "unique_USER_NUM": mapping["USER_NUM"].nunique(dropna=True),
    "duplicated_USER_KEY_rows": int(mapping["USER_KEY"].duplicated(keep=False).sum()),
    "duplicated_USER_NUM_rows": int(mapping["USER_NUM"].duplicated(keep=False).sum()),
}])
display(mapping_summary)

mapping_user_counts = (
    mapping.groupby("USER_KEY")["USER_NUM"]
    .nunique()
    .value_counts()
    .sort_index()
    .rename_axis("n_USER_NUM_per_USER_KEY")
    .reset_index(name="n_USER_KEY")
)
display(mapping_user_counts)

,rows,unique_USER_KEY,unique_USER_NUM,duplicated_USER_KEY_rows,duplicated_USER_NUM_rows
0,19877,19828,19877,97,0


,n_USER_NUM_per_USER_KEY,n_USER_KEY
0,1,19780
1,2,47
2,3,1


## 1-10. View_History 기본 구조 확인

In [33]:
required_view_cols = ["USER_NUM", "MOVIE_NUM", "watch_day", "watch_time"]
missing_cols = [c for c in required_view_cols if c not in view_check.columns]
if missing_cols:
    raise KeyError(f"View_History 필수 컬럼이 없습니다: {missing_cols}")

view_check["watch_day"] = pd.to_datetime(view_check["watch_day"], errors="coerce")
view_check["watch_time"] = pd.to_numeric(view_check["watch_time"], errors="coerce")

view_summary = pd.DataFrame([{
    "rows": len(view_check),
    "unique_USER_NUM": view_check["USER_NUM"].nunique(dropna=True),
    "unique_MOVIE_NUM": view_check["MOVIE_NUM"].nunique(dropna=True),
    "watch_day_min": view_check["watch_day"].min(),
    "watch_day_max": view_check["watch_day"].max(),
    "watch_time_missing": int(view_check["watch_time"].isna().sum()),
    "watch_time_sum": float(view_check["watch_time"].sum()),
}])
display(view_summary)

,rows,unique_USER_NUM,unique_MOVIE_NUM,watch_day_min,watch_day_max,watch_time_missing,watch_time_sum
0,106205,14892,5196,1970-01-01 00:00:00.020210301,1970-01-01 00:00:00.020210405,0,4722737.0


## 1-11. Membership, Mapping, View_History 연결 가능성

In [34]:
membership_mapping_check = membership[["USER_KEY"]].merge(mapping, on="USER_KEY", how="left", indicator=True)

membership_mapping_summary = pd.DataFrame([{
    "membership_rows": len(membership),
    "after_USER_KEY_mapping_rows": len(membership_mapping_check),
    "membership_rows_without_USER_NUM": int((membership_mapping_check["_merge"] == "left_only").sum()),
    "mapped_rows": int((membership_mapping_check["_merge"] == "both").sum()),
}])
display(membership_mapping_summary)

mapped_user_nums = set(mapping["USER_NUM"].dropna())
view_user_nums = set(view_check["USER_NUM"].dropna())

key_connection_summary = pd.DataFrame([{
    "mapping_USER_NUM_unique": len(mapped_user_nums),
    "view_USER_NUM_unique": len(view_user_nums),
    "intersection_USER_NUM": len(mapped_user_nums & view_user_nums),
    "view_USER_NUM_not_in_mapping": len(view_user_nums - mapped_user_nums),
}])
display(key_connection_summary)

,membership_rows,after_USER_KEY_mapping_rows,membership_rows_without_USER_NUM,mapped_rows
0,17876,17980,0,17980


,mapping_USER_NUM_unique,view_USER_NUM_unique,intersection_USER_NUM,view_USER_NUM_not_in_mapping
0,19877,14892,14892,0


## 1-12. Movie_Master와 View_History 영화 key 연결 확인

In [35]:
if "MOVIE_NUM" not in movie_master_check.columns:
    raise KeyError("Movie_Master에서 MOVIE_NUM alias를 만들 수 없습니다. MOVIE_ID 또는 MOVIE_NUM 컬럼을 확인하세요.")

movie_master_check["MOVIE_NUM"] = movie_master_check["MOVIE_NUM"].astype(str)
view_movie_nums = view_check["MOVIE_NUM"].astype(str)
master_movie_nums = set(movie_master_check["MOVIE_NUM"].dropna())
view_movie_set = set(view_movie_nums.dropna())

movie_key_summary = pd.DataFrame([{
    "movie_master_rows": len(movie_master_check),
    "movie_master_unique_MOVIE_NUM": len(master_movie_nums),
    "view_unique_MOVIE_NUM": len(view_movie_set),
    "view_movies_in_master": len(view_movie_set & master_movie_nums),
    "view_movies_not_in_master": len(view_movie_set - master_movie_nums),
}])
display(movie_key_summary)

,movie_master_rows,movie_master_unique_MOVIE_NUM,view_unique_MOVIE_NUM,view_movies_in_master,view_movies_not_in_master
0,14018,14018,5196,5196,0


## 1-13. 영화 메타데이터 커버리지 예비 점검

01번에서는 정확한 영화 메타데이터 통합을 수행하지 않습니다.  
다만 Wavve/KOBIS 파일이 Movie_Master와 어느 정도 연결될 수 있는지 예비 커버리지만 확인합니다.  
정식 통합은 03번에서 수행합니다.

In [36]:
def normalize_title(value) -> str:
    if pd.isna(value):
        return ""
    text = str(value).strip().lower()
    text = re.sub(r"\s+", "", text)
    text = re.sub(r"[\[\]{}<>]", "", text)
    return text

movie_for_coverage = movie_master_check.copy()
if "movie_title" not in movie_for_coverage.columns:
    raise KeyError("Movie_Master에 movie_title alias를 만들 수 없습니다. TITLE 또는 movie_title 컬럼을 확인하세요.")
movie_for_coverage["title_key"] = movie_for_coverage["movie_title"].map(normalize_title)

wavve_keys = set()
if wavve is not None:
    wavve_title_col = None
    for c in ["query_title", "title", "TITLE", "movie_title"]:
        if c in wavve.columns:
            wavve_title_col = c
            break
    if wavve_title_col:
        wavve_keys = set(wavve[wavve_title_col].map(normalize_title).dropna())
        print("Wavve title column:", wavve_title_col)
    else:
        print("Wavve title column not found. Coverage by Wavve skipped.")
else:
    print("Wavve metadata file not loaded.")

kobis_keys = set()
if kobis is not None:
    kobis_title_col = None
    for c in ["wavve_title", "query_title", "movieNm", "movieNm(api)", "title", "TITLE"]:
        if c in kobis.columns:
            kobis_title_col = c
            break
    if kobis_title_col:
        kobis_keys = set(kobis[kobis_title_col].map(normalize_title).dropna())
        print("KOBIS title column:", kobis_title_col)
    else:
        print("KOBIS title column not found. Coverage by KOBIS skipped.")
else:
    print("KOBIS metadata file not loaded.")

movie_for_coverage["covered_by_wavve"] = movie_for_coverage["title_key"].isin(wavve_keys)
movie_for_coverage["covered_by_kobis"] = movie_for_coverage["title_key"].isin(kobis_keys)
movie_for_coverage["covered_by_any"] = movie_for_coverage["covered_by_wavve"] | movie_for_coverage["covered_by_kobis"]

view_movie_table = pd.DataFrame({"MOVIE_NUM": sorted(view_movie_set)})
view_movies_for_coverage = view_movie_table.merge(
    movie_for_coverage[["MOVIE_NUM", "movie_title", "covered_by_wavve", "covered_by_kobis", "covered_by_any"]],
    on="MOVIE_NUM",
    how="left",
)

coverage_summary = pd.DataFrame([
    {
        "scope": "Movie_Master_all",
        "total_movies": len(movie_for_coverage),
        "wavve_covered": int(movie_for_coverage["covered_by_wavve"].sum()),
        "kobis_covered": int(movie_for_coverage["covered_by_kobis"].sum()),
        "any_covered": int(movie_for_coverage["covered_by_any"].sum()),
        "missing": int((~movie_for_coverage["covered_by_any"]).sum()),
        "coverage_rate": float(movie_for_coverage["covered_by_any"].mean()),
    },
    {
        "scope": "View_History_movies_only",
        "total_movies": len(view_movies_for_coverage),
        "wavve_covered": int(view_movies_for_coverage["covered_by_wavve"].fillna(False).sum()),
        "kobis_covered": int(view_movies_for_coverage["covered_by_kobis"].fillna(False).sum()),
        "any_covered": int(view_movies_for_coverage["covered_by_any"].fillna(False).sum()),
        "missing": int((~view_movies_for_coverage["covered_by_any"].fillna(False)).sum()),
        "coverage_rate": float(view_movies_for_coverage["covered_by_any"].fillna(False).mean()),
    },
])

display(coverage_summary)
coverage_summary.to_csv(TABLES_DIR / "01_data_overview_metadata_coverage.csv", index=False, encoding="utf-8-sig")

Wavve title column: query_title
KOBIS title column: wavve_title


,scope,total_movies,wavve_covered,kobis_covered,any_covered,missing,coverage_rate
0,Movie_Master_all,14018,3576,999,4575,9443,0.326366
1,View_History_movies_only,5196,3576,999,4575,621,0.880485


## 1-14. KOBIS 매칭 위험성 예비 확인

KOBIS 파일은 Wavve 미수집분 보완용입니다.  
동명이작 문제 때문에 제목만으로 붙이면 오매칭 위험이 있습니다.  
정식 품질 flag 처리는 03번에서 수행합니다.

In [37]:
if kobis is not None:
    candidate_cols = [c for c in ["wavve_title", "movieNm", "movieNm(api)", "prdtYear", "openDt", "genreNm", "watchGradeNm", "watchGrade"] if c in kobis.columns]
    print("KOBIS available risk-check columns:", candidate_cols)
    display(kobis[candidate_cols].head(10) if candidate_cols else kobis.head(10))
else:
    print("KOBIS 파일이 없어 예비 확인을 건너뜁니다.")

KOBIS available risk-check columns: ['wavve_title', 'movieNm(api)', 'prdtYear', 'openDt', 'genreNm', 'watchGrade']


,wavve_title,movieNm(api),prdtYear,openDt,genreNm,watchGrade
0,이퀼리브리엄,이퀼리브리엄,2002.0,20031002.0,액션|드라마|SF|스릴러,15세관람가
1,당신이잠든사이에(1995),당신이 잠든 사이에,2008.0,20080814.0,코미디,15세이상관람가
2,죽지않는인간들의밤,죽지않는 인간들의 밤,2019.0,20200929.0,코미디|스릴러,15세이상관람가
3,하이큐!!땅VS하늘,하이큐!! 땅 VS 하늘,2020.0,20200123.0,애니메이션,전체관람가
4,퍼펙트케어,퍼펙트 케어,2020.0,20210219.0,범죄|스릴러,15세이상관람가
5,스타게이트(1994),스타게이트,1994.0,19941217.0,SF|액션|어드벤처|판타지,중학생이상관람가
6,니나내나,니나 내나,2019.0,20191030.0,드라마,12세이상관람가
7,플라이트93(2006),플라이트93,2006.0,20060908.0,범죄|드라마,15세관람가
8,운봉,운봉,2020.0,NaN,액션,15세이상관람가
9,투빅맨,투빅맨,2020.0,NaN,액션|코미디,NaN


## 1-15. key check 결과 저장

In [38]:
key_checks = pd.concat(
    [
        membership_mapping_summary.assign(check_name="membership_to_mapping"),
        key_connection_summary.assign(check_name="mapping_to_view"),
        movie_key_summary.assign(check_name="view_to_movie_master"),
    ],
    ignore_index=True,
    sort=False,
)

display(key_checks)
key_checks.to_csv(TABLES_DIR / "01_data_overview_key_checks.csv", index=False, encoding="utf-8-sig")

,membership_rows,after_USER_KEY_mapping_rows,membership_rows_without_USER_NUM,mapped_rows,check_name,mapping_USER_NUM_unique,view_USER_NUM_unique,intersection_USER_NUM,view_USER_NUM_not_in_mapping,movie_master_rows,movie_master_unique_MOVIE_NUM,view_unique_MOVIE_NUM,view_movies_in_master,view_movies_not_in_master
0,17876.0,17980.0,0.0,17980.0,membership_to_mapping,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,NaN,NaN,NaN,NaN,mapping_to_view,19877.0,14892.0,14892.0,0.0,NaN,NaN,NaN,NaN,NaN
2,NaN,NaN,NaN,NaN,view_to_movie_master,NaN,NaN,NaN,NaN,14018.0,14018.0,5196.0,5196.0,0.0


## 1-16. 01번 결론

01번에서는 팀 합의 기준 v1 데이터의 구조를 확인했습니다.

확인해야 할 핵심 포인트는 다음입니다.

1. v1 파일 4종이 같은 폴더에서 정상 로딩되는가.
2. `Membership_v1.csv`의 target인 `is_repurchase`가 정상적으로 존재하는가.
3. `price == 100`과 `is_promotion == 1`의 관계가 유지되는가.
4. 더미 이상치 후보가 남아 있는지, 남아 있다면 02번에서 어떻게 처리할지 명시해야 한다.
5. `USER_KEY -> USER_NUM` 연결에서 복수 USER_NUM 문제가 존재하는지 확인해야 한다.
6. 분석 단위는 개인 단위가 아니라 구독 이벤트 단위로 명시해야 한다.
7. 영화 메타데이터는 03번에서 Movie_Master 기준으로 통합해야 한다.

다음 단계인 02번에서는 이 점검 결과를 바탕으로 실제 전처리 정책을 적용합니다.